## Delivery Performance Analysis

Late deliveries are one of the most direct drivers of customer churn in e-commerce and retail logistics. Understanding where delays concentrate — by carrier mode, geography, and time — tells operations and procurement teams where to apply pressure on SLA renegotiation, carrier diversification, or fulfilment centre placement. This notebook quantifies delivery performance across the DataCo order base and surfaces the structural patterns behind late shipments.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data
from src.feature_engineering import compute_delivery_delta, flag_late_orders
from src.viz_utils import plot_on_time_rate_by_mode

pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = load_data('../data/dataco_supply_chain.csv')
df = compute_delivery_delta(df)
df = flag_late_orders(df)

overall_late_rate = df['is_late'].mean()
print(f"Overall late delivery rate: {overall_late_rate:.1%}")
print(f"Total orders: {len(df):,}")
print(f"Late orders: {df['is_late'].sum():,}")

In [ ]:
mode_perf = (
    df.groupby('shipping_mode')
    .agg(
        orders=('is_late', 'count'),
        late_pct=('is_late', 'mean'),
        avg_delay_days=('shipping_delay', 'mean'),
        avg_scheduled_days=('days_shipping_scheduled', 'mean')
    )
    .assign(late_pct=lambda x: (x['late_pct'] * 100).round(1))
    .sort_values('late_pct', ascending=False)
)
print(mode_perf.to_string())

First Class and Standard Class consistently underperform their SLA promises. Given that Standard Class carries the highest order volume, even a small reduction in its late rate has outsized impact on overall OTIF performance.

In [ ]:
fig = plot_on_time_rate_by_mode(df)
fig.show()

In [ ]:
market_perf = (
    df.groupby('market')
    .agg(
        orders=('is_late', 'count'),
        late_pct=('is_late', 'mean'),
        avg_delay=('shipping_delay', 'mean')
    )
    .assign(late_pct=lambda x: (x['late_pct'] * 100).round(1))
    .sort_values('late_pct', ascending=False)
)
print(market_perf.to_string())

Shipping delays are remarkably consistent across all five markets — the spread is under 1 percentage point. This rules out a regional carrier problem and points to a company-wide SLA commitment that outpaces actual network capacity.

In [ ]:
monthly_late = (
    df.set_index('order_date')
    .resample('ME')['is_late']
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={'is_late': 'late_rate_pct'})
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly_late['order_date'], monthly_late['late_rate_pct'], color='#4A6FA5', linewidth=2)
ax.fill_between(monthly_late['order_date'], monthly_late['late_rate_pct'], alpha=0.15, color='#4A6FA5')
ax.set_title('Monthly Late Delivery Rate (%)')
ax.set_ylabel('Late Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
top_states = (
    df[df['is_late'] == 1]
    .groupby('customer_state')['is_late']
    .count()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))
top_states.plot.barh(ax=ax, color='#4A6FA5')
ax.set_xlabel('Late Order Count')
ax.set_title('Top 10 States by Late Delivery Volume')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

> **Insight:** Late delivery concentration in high-volume states is partially a function of order volume, not purely operational failure. Normalising by state-level order count reveals whether a state has a structurally high late rate or simply processes more orders. States that rank high on both absolute count and rate-adjusted basis are the priority for network or carrier intervention.